# 03 — Robustez al atlas de parcelado: conectoma humano (Budapest/HCP)

**Pregunta que se probó:** El resultado de anti-centralidad en 118 cerebros
humanos (atlas AAL-116) — ¿es un artefacto de ese atlas específico, o se
sostiene con un parcelado independiente?

**Datos:** Budapest Reference Connectome 3.0 (Szalkai et al., *Neurosci.
Lett.* 595, 2015), 477 sujetos, atlas HCP — completamente distinto del AAL.
Se usaron únicamente los archivos de tipo *edges* del repositorio.

**Resultado:** SOBREVIVIÓ, con matiz importante. Spearman(grado, V) = −1.000 ±
0.000 en las 9 particiones probadas (todos/mujeres/hombres × 3 umbrales de
consenso) — el resultado del atlas AAL queda blindado. Pero τ̃ (no V) se
rompe e invierte de signo en las particiones de umbral más permisivo
(20k), revelando que τ̃ tiene un dominio de validez limitado por el rango
espectral λ_max/λ₂ de la red.

Ver detalle completo en `paper/SPG_final_v9.docx`, Sección 5.4 y Sección 6.


## Fuentes de datos

- **Budapest Reference Connectome 3.0**
  https://networks.skewed.de/net/budapest_connectome


> **Nota:** de cada dataset se usó únicamente el archivo de tipo `edges` (lista de aristas). No se usaron archivos de nodos ni de metadatos adicionales de Netzschleuder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd, networkx as nx, numpy as np, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

class SPG:
    def __init__(self, A):
        A = np.array(A, float); np.fill_diagonal(A, 0)
        N = A.shape[0]; deg = A.sum(1)
        ev, evec = eigh(np.diag(deg) - A)
        k0 = next((k for k in range(1, N) if ev[k] > 1e-8), None)
        if k0 is None: raise ValueError("desconectado")
        M1 = np.zeros(N); M2 = np.zeros(N)
        for k in range(k0, N):
            if ev[k] < 1e-10: continue
            v2 = evec[:, k]**2
            M1 += v2/ev[k]; M2 += v2/ev[k]**2
        self.V = M1
        self.tau = np.where(M1>1e-14, M2/M1, 0)
        self.tau_tilde = ev[k0]*self.tau
        self.degree = deg; self.N = N; self.lambda2 = ev[k0]

def medir(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None)
    df = df.iloc[:, :2]; df.columns = ['s','t']
    G = nx.Graph(); G.add_edges_from(df[['s','t']].values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); s = SPG(A)
    N = s.N; deg = s.degree
    rv  = spearmanr(deg, s.V)[0]
    rtt = spearmanr(deg, s.tau_tilde)[0]
    dens = A.sum()/(N*(N-1)); cv = deg.std()/deg.mean()
    fiable = "SI" if (not np.isnan(rv) and not np.isnan(rtt)
                      and np.sign(rv)==np.sign(rtt)) else "NO/indefinido"
    print(f"{nombre:28s} {N:5d} {dens:6.3f} {cv:5.2f} {rv:+7.3f} {rtt:+7.3f} {fiable:>12s}")
    return dict(red=nombre, N=N, dens=dens, cv=cv, sp_V=rv, sp_tt=rtt, fiable=fiable)

print("BLOQUE BASE cargado. Listo para medir.")


BLOQUE BASE cargado. Listo para medir.


https://networks.skewed.de/net/budapest_connectome

----------------------------------------------------------------------
Eng
For this project we use budapest_connectome — Budapest Reference Connectome 3.0. We used the CSV files in the folder, working exclusively with the files named "edges."

The pattern used was:
df = pd.read_csv(archivo, comment='#', header=None).iloc[:,:2]
df.columns = ['s','t']
G = nx.Graph(); G.add_edges_from(df.values)

That is, each row of the CSV is a pair (source_node, target_node) representing a connection.

----------------------------------------------------------------------
Esp
Para este proyecto vamos a usar budapest_connectome — Budapest Reference Connectome 3.0 utilizaroms los CSV en la carpeta utilizamos unicamente los archivos con nombre (edges)

el patron que se uso es
df = pd.read_csv(archivo, comment='#', header=None).iloc[:,:2]
df.columns = ['s','t']
G = nx.Graph(); G.add_edges_from(df.values)

Es decir, cada fila del CSV es un par (nodo_origen, nodo_destino) que representa una conexión

In [ ]:
CARPETA = './data'   # coloca aquí tus archivos CSV descargados  # ajusta a tu ruta
import os

print("archivos:")
for f in sorted(os.listdir(CARPETA)):
    if f.endswith('.csv'): print("  ", f)
print("="*60)

res_budapest = []
print(f"{'red':28s} {'N':>5s} {'dens':>6s} {'CV':>5s} {'Sp(V)':>7s} {'Sp(tt)':>7s} {'fiable':>12s}")
print("-"*76)
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    try:
        res_budapest.append(medir(os.path.join(CARPETA, f), f[:-4]))
    except Exception as e:
        print(f"{f[:-4]:28s} ERROR: {str(e)[:40]}")

# síntesis si hay varias versiones
if res_budapest:
    import numpy as np
    stt = [r['sp_tt'] for r in res_budapest]
    sv  = [r['sp_V']  for r in res_budapest]
    print(f"\n=== SÍNTESIS budapest (n={len(stt)}) ===")
    print(f"Sp(k,tt): {np.mean(stt):+.3f} ± {np.std(stt):.3f}")
    print(f"Sp(k,V):  {np.mean(sv):+.3f} ± {np.std(sv):.3f}")
    print(f"% negativos: {100*np.mean(np.array(stt)<0):.0f}%")


archivos:
   all_1m.csv
   all_200k.csv
   all_20k.csv
   female_1m.csv
   female_200k.csv
   female_20k.csv
   male_1m.csv
   male_200k.csv
   male_20k.csv
red                              N   dens    CV   Sp(V)  Sp(tt)       fiable
----------------------------------------------------------------------------
all_1m                        1015  0.235  0.42  -1.000  -0.985           SI
all_200k                      1015  0.203  0.45  -1.000  -0.949           SI
all_20k                       1015  0.137  0.51  -1.000  +0.398 NO/indefinido
female_1m                     1015  0.217  0.45  -1.000  -0.979           SI
female_200k                   1015  0.186  0.48  -1.000  -0.924           SI
female_20k                    1015  0.122  0.54  -1.000  +0.502 NO/indefinido
male_1m                       1015  0.180  0.45  -1.000  -0.848           SI
male_200k                     1015  0.154  0.48  -1.000  -0.391           SI
male_20k                      1015  0.104  0.54  -1.000  +0.770 NO/inde

In [ ]:
import pandas as pd, numpy as np, networkx as nx
from scipy.linalg import eigh

def diagnostico(archivo, nombre):
    df = pd.read_csv(archivo, comment='#', header=None).iloc[:,:2]
    df.columns=['s','t']
    G = nx.Graph(); G.add_edges_from(df.values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); N=A.shape[0]
    ev = eigh(np.diag(A.sum(1))-A, eigvals_only=True)
    k0 = next(k for k in range(1,N) if ev[k]>1e-8)
    print(f"\n=== {nombre} ===")
    print(f"  N={N}  densidad={A.sum()/(N*(N-1)):.3f}")
    print(f"  lambda_2 = {ev[k0]:.6e}")
    print(f"  lambda_3 = {ev[k0+1]:.6e}")
    print(f"  gap lambda_2/lambda_3 = {ev[k0]/ev[k0+1]:.4f}")
    print(f"  lambda_max = {ev[-1]:.2f}")
    print(f"  rango lambda_max/lambda_2 = {ev[-1]/ev[k0]:.1f}")
    print(f"  autovalores < 1e-6: {(np.abs(ev)<1e-6).sum()}")
    # cuántos nodos de grado muy bajo (periféricos)
    deg = A.sum(1)
    print(f"  nodos grado 1: {(deg==1).sum()}   grado<=3: {(deg<=3).sum()}")

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/Budapest'   # ajusta
import os
diagnostico(os.path.join(CARPETA,'all_20k.csv'), 'all_20k (FALLA)')
diagnostico(os.path.join(CARPETA,'all_1m.csv'),  'all_1m (FUNCIONA)')



=== all_20k (FALLA) ===
  N=1015  densidad=0.137
  lambda_2 = 2.653738e+00
  lambda_3 = 6.681274e+00
  gap lambda_2/lambda_3 = 0.3972
  lambda_max = 467.17
  rango lambda_max/lambda_2 = 176.0
  autovalores < 1e-6: 1
  nodos grado 1: 0   grado<=3: 0

=== all_1m (FUNCIONA) ===
  N=1015  densidad=0.235
  lambda_2 = 1.340748e+01
  lambda_3 = 1.671536e+01
  gap lambda_2/lambda_3 = 0.8021
  lambda_max = 693.11
  rango lambda_max/lambda_2 = 51.7
  autovalores < 1e-6: 1
  nodos grado 1: 0   grado<=3: 0


In [ ]:
import pandas as pd, numpy as np, networkx as nx, os
from scipy.linalg import eigh
from scipy.stats import spearmanr

CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/Budapest'   # ajusta si es otra

filas = []
for f in sorted(os.listdir(CARPETA)):
    if not f.endswith('.csv'): continue
    df = pd.read_csv(os.path.join(CARPETA,f), comment='#', header=None).iloc[:,:2]
    df.columns=['s','t']
    G = nx.Graph(); G.add_edges_from(df.values)
    G.remove_edges_from(nx.selfloop_edges(G))
    G = G.subgraph(max(nx.connected_components(G),key=len)).copy()
    A = nx.to_numpy_array(G); N=A.shape[0]; deg=A.sum(1)
    ev, evec = eigh(np.diag(deg)-A)
    k0 = next(k for k in range(1,N) if ev[k]>1e-8)
    lam2=ev[k0]
    gap = ev[k0]/ev[k0+1]
    rango = ev[-1]/ev[k0]
    # V y tau_tilde
    M1=np.zeros(N); M2=np.zeros(N)
    for k in range(k0,N):
        v2=evec[:,k]**2; M1+=v2/ev[k]; M2+=v2/ev[k]**2
    V=M1; tt=lam2*np.where(M1>1e-14,M2/M1,0)
    spV = spearmanr(deg,V)[0]; sptt = spearmanr(deg,tt)[0]
    diverg = abs(spV - sptt)
    filas.append((f[:-4], gap, rango, spV, sptt, diverg))
    print(f"{f[:-4]:14s} gap={gap:.3f} rango={rango:6.1f} "
          f"Sp(V)={spV:+.3f} Sp(tt)={sptt:+.3f} |div|={diverg:.3f}")

# correlación: ¿el gap predice la divergencia?
import numpy as np
gaps = [r[1] for r in filas]; divs=[r[5] for r in filas]
rho = spearmanr(gaps, divs)[0]
print(f"\n>>> Spearman(gap, divergencia) = {rho:+.3f}")
print(">>> Si es NEGATIVO fuerte: gap bajo -> mas divergencia. CONFIRMA el mecanismo.")


all_1m         gap=0.802 rango=  51.7 Sp(V)=-1.000 Sp(tt)=-0.985 |div|=0.015
all_200k       gap=0.578 rango=  69.7 Sp(V)=-1.000 Sp(tt)=-0.949 |div|=0.051
all_20k        gap=0.397 rango= 176.0 Sp(V)=-1.000 Sp(tt)=+0.398 |div|=1.398
female_1m      gap=0.735 rango=  59.4 Sp(V)=-1.000 Sp(tt)=-0.979 |div|=0.021
female_200k    gap=0.530 rango=  78.7 Sp(V)=-1.000 Sp(tt)=-0.924 |div|=0.076
female_20k     gap=0.542 rango= 214.0 Sp(V)=-1.000 Sp(tt)=+0.502 |div|=1.502
male_1m        gap=0.580 rango=  82.8 Sp(V)=-1.000 Sp(tt)=-0.848 |div|=0.151
male_200k      gap=0.415 rango= 116.6 Sp(V)=-1.000 Sp(tt)=-0.391 |div|=0.609
male_20k       gap=0.311 rango= 276.6 Sp(V)=-1.000 Sp(tt)=+0.770 |div|=1.769

>>> Spearman(gap, divergencia) = -0.833
>>> Si es NEGATIVO fuerte: gap bajo -> mas divergencia. CONFIRMA el mecanismo.


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr

# reusa tu clase SPG y carga UNA red confiable (celegans_metabolic o fly_larva)
CARPETA = '/content/drive/MyDrive/PYTHON/C.Elegans CSV/CSV'
import pandas as pd, os
df = pd.read_csv(os.path.join(CARPETA,'Metabolism.csv'), comment='#', header=None).iloc[:,:2]
df.columns=['s','t']
G = nx.Graph(); G.add_edges_from(df.values)
G.remove_edges_from(nx.selfloop_edges(G))
G = G.subgraph(max(nx.connected_components(G),key=len)).copy()

s = SPG(nx.to_numpy_array(G))
V = s.V
nodes = list(G.nodes())
def arr(d): return np.array([d[n] for n in nodes])

deg = dict(G.degree())
bet = nx.betweenness_centrality(G)
eig = nx.eigenvector_centrality_numpy(G)
clo = nx.closeness_centrality(G)
cfc = nx.current_flow_closeness_centrality(G)

print("Spearman(V, centralidad):")
for name,c in [('degree',deg),('betweenness',bet),('eigenvector',eig),
               ('closeness',clo),('current_flow',cfc)]:
    print(f"  V vs {name:14s}: {spearmanr(V,arr(c))[0]:+.3f}")


Spearman(V, centralidad):
  V vs degree        : -0.962
  V vs betweenness   : -0.749
  V vs eigenvector   : -0.854
  V vs closeness     : -0.651
  V vs current_flow  : -1.000


In [ ]:
# en una red con λmax/λ2 BAJO (donde tau_tilde es válido), p.ej. all_1m de budapest
s = SPG(A)
from scipy.stats import spearmanr
# ¿tau_tilde y V ordenan igual los nodos?
rho = spearmanr(s.V, s.tau_tilde)[0]
print(f"Spearman(V, tau_tilde) entre nodos = {rho:.4f}")
# si ~1.0 -> tau_tilde NO aporta nada que V no tenga
# si <0.9 -> tau_tilde captura algo distinto; hay que ver QUÉ


Spearman(V, tau_tilde) entre nodos = -0.7597


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr

# G ya cargado (la red válida, all_1m u otra con lambda_max/lambda2 bajo)
s = SPG(nx.to_numpy_array(G))
nodes = list(G.nodes())
def arr(d): return np.array([d[n] for n in nodes])

props = {
    'degree':        arr(dict(G.degree())),
    'clustering':    arr(nx.clustering(G)),
    'betweenness':   arr(nx.betweenness_centrality(G)),
    'core_number':   arr(nx.core_number(G)),
    'avg_neigh_deg': arr(nx.average_neighbor_degree(G)),
}

print(f"{'propiedad':16s} {'V':>8s} {'tau_tilde':>10s}  {'difieren?'}")
print("-"*50)
for name, x in props.items():
    rv  = spearmanr(s.V, x)[0]
    rtt = spearmanr(s.tau_tilde, x)[0]
    dif = "SÍ" if abs(rv-rtt) > 0.3 else "no"
    print(f"{name:16s} {rv:+8.3f} {rtt:+10.3f}  {dif}")


propiedad               V  tau_tilde  difieren?
--------------------------------------------------
degree             -0.962     -0.844  no
clustering         +0.619     +0.465  no
betweenness        -0.749     -0.577  no
core_number        -0.961     -0.914  no
avg_neigh_deg      +0.054     -0.077  no


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr
import networkx.algorithms.community as nxcom

def prueba_tau_vs_V(G, nombre):
    s = SPG(nx.to_numpy_array(G))
    nodes = list(G.nodes()); idx = {n:i for i,n in enumerate(nodes)}
    V, tt = s.V, s.tau_tilde
    print(f"\n=== {nombre} (N={len(nodes)}) ===")

    # detectar comunidades (Louvain)
    try:
        comms = nxcom.louvain_communities(G, seed=42)
        comm_of = {n:i for i,c in enumerate(comms) for n in c}
    except Exception as e:
        print("  no se pudo comunidad:", e); return

    # TAREA 1: participation coefficient (rol de conector entre módulos)
    #   nodo conector = conecta a muchas comunidades distintas
    part = []
    for n in nodes:
        degs = {}
        for nb in G.neighbors(n):
            c = comm_of[nb]; degs[c] = degs.get(c,0)+1
        k = G.degree(n)
        P = 1 - sum((d/k)**2 for d in degs.values()) if k>0 else 0
        part.append(P)
    part = np.array(part)

    # TAREA 2: within-module degree (hub de módulo)
    wmd = []
    for n in nodes:
        c = comm_of[n]
        same = [m for m in comms[c]]
        sub = G.subgraph(same)
        wmd.append(sub.degree(n) if n in sub else 0)
    wmd = np.array(wmd, float)

    # TAREA 3: es nodo frontera (tiene vecinos en otra comunidad)?
    frontera = np.array([1 if any(comm_of[nb]!=comm_of[n] for nb in G.neighbors(n)) else 0
                         for n in nodes], float)

    for tarea, y in [('participation', part), ('within_mod_deg', wmd), ('frontera', frontera)]:
        rV  = abs(spearmanr(V, y)[0])
        rtt = abs(spearmanr(tt, y)[0])
        gana = "  <-- TAU GANA" if rtt - rV > 0.15 else ""
        print(f"  {tarea:16s}  |V|={rV:.3f}  |tau|={rtt:.3f}{gana}")

# corre en 3-4 redes CON estructura de comunidades clara y lambda_max/lambda2 BAJO
# (importante: solo redes donde tau_tilde es válido)
# ejemplos: jazz_collab, celegans_metabolic, una budapest _1m
prueba_tau_vs_V(G, "tu_red")



=== tu_red (N=453) ===
  participation     |V|=0.669  |tau|=0.660
  within_mod_deg    |V|=0.670  |tau|=0.521
  frontera          |V|=0.601  |tau|=0.613


In [ ]:
import networkx as nx, numpy as np
from scipy.stats import spearmanr, pearsonr

s = SPG(nx.to_numpy_array(G))
V = s.V
nodes = list(G.nodes())

# L+ diagonal directa (pseudoinversa)
L = nx.laplacian_matrix(G).toarray().astype(float)
Lplus = np.linalg.pinv(L)
Lplus_diag = np.diag(Lplus)

# current-flow closeness de networkx
cfc = np.array([nx.current_flow_closeness_centrality(G)[n] for n in nodes])

print("¿Tu V es EXACTAMENTE L+_ii?")
print(f"  Pearson(V, L+_ii)  = {pearsonr(V, Lplus_diag)[0]:.10f}")
print(f"  máx diferencia abs = {np.max(np.abs(V - Lplus_diag)):.2e}")
print(f"  ¿idénticas? {np.allclose(V, Lplus_diag)}")
print()
print("¿Tu V es EXACTAMENTE current-flow closeness?")
print(f"  Pearson(V, CFC)    = {pearsonr(V, cfc)[0]:.6f}")
print(f"  Spearman(V, CFC)   = {spearmanr(V, cfc)[0]:.6f}")
print(f"  ¿idénticas? {np.allclose(V, cfc)}")
print(f"  ¿idénticas tras escalar? {np.allclose(V/V.mean(), cfc/cfc.mean())}")


¿Tu V es EXACTAMENTE L+_ii?
  Pearson(V, L+_ii)  = 1.0000000000
  máx diferencia abs = 2.62e-14
  ¿idénticas? True

¿Tu V es EXACTAMENTE current-flow closeness?
  Pearson(V, CFC)    = -0.816861
  Spearman(V, CFC)   = -0.999997
  ¿idénticas? False
  ¿idénticas tras escalar? False
